# [시장분석] 강남3구 아파트, 동별 대장단지끼리 붙으면 어디가 1위일까?

2016년 7월 당시 ㎡당 KB 일반가가 가장 높았던 단지를 각 동의 대장단지로 먼저 선정하고, 2026년 7월까지 최근 10년 상승률을 비교합니다.

- 강남구·서초구·송파구 순수 아파트 중 300세대 이상 단지
- 2016년 7월과 2026년 7월 동일 단지·동일 면적 가격이 모두 있는 타입
- 전용 75~93㎡ 범위에서 단지별 84㎡에 가장 가까운 타입 사용
- 300세대 이상 유효 단지가 2개 이상인 동만 분석


In [ ]:
# @title 강남3구 동별 대장단지의 최근 10년 상승률을 계산하세요
"""2016년 당시 강남3구 동별 대장단지의 최근 10년 상승률을 비교한다.

KB부동산 매매 일반가를 사용한다. 동일 단지·동일 면적의 2016년 7월과
2026년 7월 가격을 매칭하고, 타입별 ㎡당 가격의 중앙값을 단지 대표값으로
사용한다. 각 동에서 2016년 ㎡당 가격이 가장 높은 단지를 당시 대장단지로
선정한다. 전용 75~93㎡ 범위에서 84㎡에 가장 가까운 타입을 사용하며,
300세대 미만과 주상복합을 제외하고 유효 단지가 2개 이상인
동만 분석한다.
"""

from __future__ import annotations

import os
from html import escape
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-an-gangnam3-flagship")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import FileLink, HTML, Image, Markdown, display
from matplotlib import font_manager
from matplotlib.offsetbox import AnnotationBbox, HPacker, TextArea


START_YEAR_MONTH = "2016년 7월"
END_YEAR_MONTH = "2026년 7월"
COMPARISON_YEARS = 10
MIN_HOUSEHOLDS = 300
MIN_COMPLEX_COUNT = 2
DONG_MIN_HOUSEHOLDS = 100
DONG_MIN_COMPLEX_COUNT = 3
MIN_PRIVATE_AREA_SQM = 75.0
MAX_PRIVATE_AREA_SQM = 93.0
TARGET_PRIVATE_AREA_SQM = 84.0

IS_COLAB = bool(os.environ.get("COLAB_RELEASE_TAG")) or Path("/content").exists()
OUTPUT_DIR = Path("/content/output") if IS_COLAB else Path("output")
MATCHED_TYPE_PATH = OUTPUT_DIR / "kb_gangnam3_matched_types_201607_202607.csv"
RESULT_PATH = OUTPUT_DIR / "gangnam3_dong_flagship_10y_growth.csv"
CHART_PATH = OUTPUT_DIR / "gangnam3_dong_flagship_10y_growth.png"
COMPLETION_YEAR_CHART_PATH = (
    OUTPUT_DIR / "gangnam3_dong_flagship_10y_growth_by_completion_year.png"
)

BACKGROUND_COLOR = "#FFFFFF"
GRID_COLOR = "#DEDCD6"
TEXT_COLOR = "#0B0B0B"
SECONDARY_TEXT_COLOR = "#64748B"
TICK_COLOR = "#777777"
DISTRICT_COLORS = {
    "강남구": "#2F7DD3",
    "서초구": "#1FAE7A",
    "송파구": "#F2A000",
}

BRAND_SIZE = 13
TITLE_SIZE = 21
SUBTITLE_SIZE = 16
AXIS_TITLE_SIZE = 15
TICK_SIZE = 15
LEGEND_SIZE = 15
DATA_LABEL_SIZE = 14
FOOTNOTE_SIZE = 13


def configure_font() -> str:
    """설치된 한글 글꼴을 Matplotlib 기본 글꼴로 지정한다."""
    candidates = (
        "Pretendard",
        "Apple SD Gothic Neo",
        "Noto Sans CJK KR",
        "Malgun Gothic",
    )
    installed = {font.name for font in font_manager.fontManager.ttflist}
    selected = next(
        (name for name in candidates if name in installed), "DejaVu Sans"
    )
    plt.rcParams.update({"font.family": selected, "axes.unicode_minus": False})
    return selected


def load_matched_types(path: Path) -> pd.DataFrame:
    """저장된 동일 단지·동일 면적 매칭 자료를 읽고 검증한다."""
    if not path.exists():
        raise FileNotFoundError(f"저장된 매칭 자료가 없습니다: {path}")
    data = pd.read_csv(path)
    required = {
        "자치구",
        "동",
        "단지기본일련번호",
        "아파트",
        "전용면적_㎡",
        "시작가격_만원",
        "종료가격_만원",
        "준공연도",
        "세대수",
        "면적일련번호",
    }
    missing = required.difference(data.columns)
    if missing:
        raise ValueError(f"매칭 자료에 필요한 열이 없습니다: {sorted(missing)}")
    return data


def calculate_flagships(
    matched_types: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """2016년 ㎡당 가격 1위 단지와 10년 상승률을 동별로 계산한다."""
    valid_types = matched_types.loc[
        (matched_types["세대수"] >= MIN_HOUSEHOLDS)
        & matched_types["전용면적_㎡"].between(
            MIN_PRIVATE_AREA_SQM, MAX_PRIVATE_AREA_SQM
        )
        & (matched_types["시작가격_만원"] > 0)
        & (matched_types["종료가격_만원"] > 0)
    ].copy()
    valid_types["area_distance_sqm"] = (
        valid_types["전용면적_㎡"] - TARGET_PRIVATE_AREA_SQM
    ).abs()
    nearest_distance = valid_types.groupby("단지기본일련번호")[
        "area_distance_sqm"
    ].transform("min")
    valid_types = valid_types.loc[
        valid_types["area_distance_sqm"].eq(nearest_distance)
    ].copy()
    valid_types["start_price_per_sqm"] = (
        valid_types["시작가격_만원"] / valid_types["전용면적_㎡"]
    )
    valid_types["end_price_per_sqm"] = (
        valid_types["종료가격_만원"] / valid_types["전용면적_㎡"]
    )

    group_columns = [
        "자치구",
        "동",
        "단지기본일련번호",
        "아파트",
        "준공연도",
        "세대수",
    ]
    complexes = (
        valid_types.groupby(group_columns, as_index=False)
        .agg(
            valid_type_count=("면적일련번호", "nunique"),
            representative_area_sqm=("전용면적_㎡", "median"),
            start_price_per_sqm=("start_price_per_sqm", "median"),
            end_price_per_sqm=("end_price_per_sqm", "median"),
        )
    )
    complexes["cumulative_growth"] = (
        complexes["end_price_per_sqm"] / complexes["start_price_per_sqm"] - 1
    )
    complexes["annual_growth"] = (
        complexes["end_price_per_sqm"] / complexes["start_price_per_sqm"]
    ) ** (1 / COMPARISON_YEARS) - 1

    dong_counts = (
        complexes.groupby(["자치구", "동"], as_index=False)
        .size()
        .rename(columns={"size": "valid_complex_count"})
    )
    eligible_dongs = dong_counts.loc[
        dong_counts["valid_complex_count"] >= MIN_COMPLEX_COUNT
    ]
    eligible_complexes = complexes.merge(
        eligible_dongs, on=["자치구", "동"], how="inner"
    )
    ranked = eligible_complexes.sort_values(
        ["자치구", "동", "start_price_per_sqm", "단지기본일련번호"],
        ascending=[True, True, False, True],
    )
    flagships = (
        ranked.groupby(["자치구", "동"], as_index=False)
        .first()
        .sort_values("annual_growth", ascending=False)
        .reset_index(drop=True)
    )
    flagships.insert(0, "순위", range(1, len(flagships) + 1))
    overall_median = flagships["annual_growth"].median()
    flagships["전체중앙값대비"] = (
        flagships["annual_growth"] - overall_median
    )
    flagships["KB부동산링크"] = flagships["단지기본일련번호"].map(
        lambda value: f"https://kbland.kr/se/c/{int(value)}"
    )
    return flagships, complexes


def calculate_rank_comparison(
    matched_types: pd.DataFrame, flagships: pd.DataFrame
) -> pd.DataFrame:
    """동 전체 상승률 순위와 대장단지 상승률 순위를 결합한다."""
    eligible_types = matched_types.loc[
        matched_types["세대수"].ge(DONG_MIN_HOUSEHOLDS)
    ].copy()
    complex_growth = (
        eligible_types.groupby(
            ["자치구", "동", "단지기본일련번호", "아파트"],
            as_index=False,
        )
        .agg(연평균상승률=("연평균상승률", "median"))
    )
    dong_growth = (
        complex_growth.groupby(["자치구", "동"], as_index=False)
        .agg(
            유효단지수=("단지기본일련번호", "nunique"),
            동연평균상승률=("연평균상승률", "median"),
        )
    )
    dong_growth = (
        dong_growth.loc[
            dong_growth["유효단지수"].ge(DONG_MIN_COMPLEX_COUNT)
        ]
        .sort_values("동연평균상승률", ascending=False)
        .reset_index(drop=True)
    )
    dong_growth.insert(0, "동전체순위", range(1, len(dong_growth) + 1))
    comparison = dong_growth.merge(
        flagships[["순위", "자치구", "동", "아파트", "KB부동산링크"]],
        on=["자치구", "동"],
        how="inner",
    ).rename(columns={"순위": "대장단지순위"})
    comparison["순위차이"] = (
        comparison["동전체순위"] - comparison["대장단지순위"]
    )
    return comparison.sort_values("동전체순위").reset_index(drop=True)


def build_criteria_markdown(
    matched_types: pd.DataFrame,
    complexes: pd.DataFrame,
    flagships: pd.DataFrame,
) -> str:
    """대장단지 선정 기준과 표본 규모를 Markdown으로 만든다."""
    eligible_complex_count = len(
        complexes.merge(
            flagships[["자치구", "동"]],
            on=["자치구", "동"],
            how="inner",
        )
    )
    return (
        "### ■ 대장단지 선정 기준\n\n"
        f"비교 기간: {START_YEAR_MONTH}~{END_YEAR_MONTH} (10년간)  \n"
        f"매칭 자료: 동일 단지·동일 면적 "
        f"{matched_types['단지기본일련번호'].nunique():,}개 단지, "
        f"{matched_types['면적일련번호'].nunique():,}개 타입  \n"
        f"분석 대상: 300세대 이상 후보 {eligible_complex_count:,}개 단지, "
        f"유효 단지 2개 이상 {len(flagships):,}개 동  \n"
        "면적 기준: 전용 75~93㎡ 중 단지별 84㎡에 가장 가까운 타입  \n"
        "선정 방식: 대표 타입의 2016년 ㎡당 KB 일반가를 구한 뒤, "
        "각 동에서 가장 높은 단지를 대장단지로 선정  \n"
        "상승률 산출: 선정된 단지의 2016년·2026년 ㎡당 가격 중앙값으로 CAGR 계산"
    )


def format_price_per_sqm(value: float) -> str:
    """㎡당 만원 가격을 정수 문자열로 표시한다."""
    return f"{value:,.0f}만원"


def build_table_html(flagships: pd.DataFrame) -> str:
    """동별 대장단지 상승률을 700px HTML 표로 만든다."""
    rows = []
    for row in flagships.itertuples(index=False):
        apartment = (
            f'<a href="{escape(row.KB부동산링크)}" target="_blank" '
            f'rel="noopener noreferrer">{escape(str(row.아파트))}</a>'
        )
        rows.append(
            "<tr>"
            f'<td class="identifier">{row.순위}</td>'
            f'<td class="identifier">{escape(str(row.자치구))}</td>'
            f'<td class="identifier">{escape(str(row.동))}</td>'
            f'<td class="row-label">{apartment}</td>'
            f'<td class="identifier">{int(row.준공연도)}</td>'
            f'<td class="number">{int(row.세대수):,}세대</td>'
            f'<td class="number">{row.representative_area_sqm:.2f}㎡</td>'
            f'<td class="number">{format_price_per_sqm(row.start_price_per_sqm)}</td>'
            f'<td class="number">{format_price_per_sqm(row.end_price_per_sqm)}</td>'
            f'<td class="number"><strong>{row.annual_growth:.2%}</strong></td>'
            "</tr>"
        )
    body = "".join(rows)
    columns = "".join("<col>" for _ in range(10))
    return f"""
<style>
.flagship-table-section {{ max-width:700px; margin:0 0 28px; font-family:Pretendard,-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif; }}
.flagship-table-section .brand {{ margin:0 0 5px; color:#64748b; font-size:13px; }}
.flagship-table-section h2 {{ margin:0 0 4px; color:#0f172a; font-size:20px; line-height:1.3; }}
.flagship-table-section .caption {{ margin:0 0 14px; color:#64748b; font-size:13px; }}
.flagship-table-wrap {{ overflow:hidden; border:1px solid #f0f2f5; border-radius:12px; }}
.flagship-table {{ width:100%; table-layout:fixed; border-collapse:separate; border-spacing:0; color:#1e293b; font-size:13px; font-variant-numeric:tabular-nums; }}
.flagship-table col:nth-child(1) {{ width:4%; }}
.flagship-table col:nth-child(2), .flagship-table col:nth-child(3) {{ width:7%; }}
.flagship-table col:nth-child(4) {{ width:22%; }}
.flagship-table col:nth-child(5) {{ width:6%; }}
.flagship-table col:nth-child(6) {{ width:8%; }}
.flagship-table col:nth-child(7) {{ width:9%; }}
.flagship-table col:nth-child(8), .flagship-table col:nth-child(9) {{ width:12%; }}
.flagship-table col:nth-child(10) {{ width:13%; }}
.flagship-table th {{ padding:11px 3px; background:#2b4a75; color:#fff; line-height:1.35; word-break:keep-all; }}
.flagship-table td {{ padding:11px 3px; border-bottom:1px solid #f0f2f5; background:#fff; line-height:1.45; white-space:nowrap; }}
.flagship-table tbody tr:nth-child(even) td {{ background:#fafbfc; }}
.flagship-table tbody tr:last-child td {{ border-bottom:0; }}
.flagship-table .identifier {{ text-align:center; }}
.flagship-table .row-label {{ text-align:left; overflow:hidden; text-overflow:ellipsis; }}
.flagship-table .number {{ text-align:right; }}
.flagship-table a {{ color:#4f8bc9; font-weight:600; text-decoration:underline; }}
.flagship-table-section .footnote {{ margin:10px 0 0; color:#64748b; font-size:13px; line-height:1.5; }}
</style>
<section class="flagship-table-section">
  <p class="brand">대도시 연구실</p>
  <h2>강남3구 동별 대장단지 상승률 | 최근 10년</h2>
  <p class="caption">전용 75~93㎡ · 84㎡ 최근접 타입 · 300세대 이상 · 2016년 7월~2026년 7월</p>
  <div class="flagship-table-wrap">
    <table class="flagship-table">
      <colgroup>{columns}</colgroup>
      <thead><tr><th>순위</th><th>자치구</th><th>동</th><th>대장단지</th><th>준공</th><th>세대수</th><th>대표<br>전용면적</th><th>2016년<br>㎡당 가격</th><th>2026년<br>㎡당 가격</th><th>연평균<br>상승률</th></tr></thead>
      <tbody>{body}</tbody>
    </table>
  </div>
  <p class="footnote">※ 전용 75~93㎡ 중 단지별 84㎡에 가장 가까운 타입 사용<br>※ 같은 거리에 복수 타입이 있으면 ㎡당 KB 일반가 중앙값으로 통합<br>※ 대장단지는 2016년 7월 가격으로 선정하며 300세대 미만·주상복합 제외<br>※ 유효 단지가 2개 이상인 동만 분석<br>※ 단지명을 누르면 KB부동산 페이지가 열림</p>
</section>
"""


def build_rank_comparison_table_html(comparison: pd.DataFrame) -> str:
    """동 전체 순위와 대장단지 순위를 비교하는 HTML 표를 만든다."""
    rows = []
    for row in comparison.itertuples(index=False):
        apartment = (
            f'<a href="{escape(row.KB부동산링크)}" target="_blank" '
            f'rel="noopener noreferrer">{escape(str(row.아파트))}</a>'
        )
        difference = int(row.순위차이)
        difference_class = (
            "positive" if difference > 0 else "negative" if difference < 0 else "neutral"
        )
        difference_text = f"{difference:+d}" if difference else "0"
        rows.append(
            "<tr>"
            f'<td class="identifier">{escape(str(row.자치구))}</td>'
            f'<td class="identifier">{escape(str(row.동))}</td>'
            f'<td class="number"><strong>{int(row.동전체순위)}</strong></td>'
            f'<td class="row-label">{apartment}</td>'
            f'<td class="number"><strong>{int(row.대장단지순위)}</strong></td>'
            f'<td class="number {difference_class}"><strong>{difference_text}</strong></td>'
            "</tr>"
        )
    body = "".join(rows)
    return f"""
<style>
.rank-comparison-section {{ max-width:700px; margin:0 0 28px; font-family:Pretendard,-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif; }}
.rank-comparison-section .brand {{ margin:0 0 5px; color:#64748b; font-size:13px; }}
.rank-comparison-section h2 {{ margin:0 0 4px; color:#0f172a; font-size:20px; line-height:1.3; }}
.rank-comparison-section .caption {{ margin:0 0 14px; color:#64748b; font-size:13px; }}
.rank-comparison-wrap {{ overflow:hidden; border:1px solid #f0f2f5; border-radius:12px; }}
.rank-comparison-table {{ width:100%; table-layout:fixed; border-collapse:separate; border-spacing:0; color:#1e293b; font-size:13px; font-variant-numeric:tabular-nums; }}
.rank-comparison-table col:nth-child(1) {{ width:12%; }}
.rank-comparison-table col:nth-child(2) {{ width:12%; }}
.rank-comparison-table col:nth-child(3) {{ width:14%; }}
.rank-comparison-table col:nth-child(4) {{ width:34%; }}
.rank-comparison-table col:nth-child(5) {{ width:15%; }}
.rank-comparison-table col:nth-child(6) {{ width:13%; }}
.rank-comparison-table th {{ padding:11px 4px; background:#2b4a75; color:#fff; line-height:1.35; word-break:keep-all; }}
.rank-comparison-table td {{ padding:11px 5px; border-bottom:1px solid #f0f2f5; background:#fff; line-height:1.45; white-space:nowrap; }}
.rank-comparison-table tbody tr:nth-child(even) td {{ background:#fafbfc; }}
.rank-comparison-table tbody tr:last-child td {{ border-bottom:0; }}
.rank-comparison-table .identifier {{ text-align:center; }}
.rank-comparison-table .row-label {{ text-align:left; overflow:hidden; text-overflow:ellipsis; }}
.rank-comparison-table .number {{ text-align:right; }}
.rank-comparison-table .positive {{ color:#2f7dd3; }}
.rank-comparison-table .negative {{ color:#f06432; }}
.rank-comparison-table .neutral {{ color:#64748b; }}
.rank-comparison-table a {{ color:#4f8bc9; font-weight:600; text-decoration:underline; }}
.rank-comparison-section .footnote {{ margin:10px 0 0; color:#64748b; font-size:13px; line-height:1.5; }}
</style>
<section class="rank-comparison-section">
  <p class="brand">대도시 연구실</p>
  <h2>강남3구 동별 상승률 순위와 대장단지 순위 비교</h2>
  <p class="caption">KB 매매 일반가 · 2016년 7월~2026년 7월</p>
  <div class="rank-comparison-wrap">
    <table class="rank-comparison-table">
      <colgroup><col><col><col><col><col><col></colgroup>
      <thead><tr><th>자치구</th><th>동</th><th>동 전체<br>순위</th><th>대장단지</th><th>대장단지<br>순위</th><th>순위 차이</th></tr></thead>
      <tbody>{body}</tbody>
    </table>
  </div>
  <p class="footnote">※ 동 전체 순위는 100세대 이상 순수 아파트의 모든 면적 기준<br>※ 대장단지 순위는 전용 75~93㎡·84㎡ 최근접 타입·300세대 이상 기준<br>※ 순위 차이 = 동 전체 순위 - 대장단지 순위 (양수는 대장단지 순위가 더 높음)</p>
</section>
"""


def create_chart(
    flagships: pd.DataFrame,
    output_path: Path,
    order_by_completion_year: bool = False,
) -> Path:
    """대장단지 상승률의 전체 중앙값 대비 차이를 막대로 저장한다."""
    font_family = configure_font()
    if order_by_completion_year:
        chart = flagships.sort_values(
            ["준공연도", "순위"], ascending=[False, False]
        ).copy()
    else:
        chart = flagships.sort_values("annual_growth", ascending=True).copy()
    values = chart["전체중앙값대비"].to_numpy() * 100
    colors = ["#F06432" if value >= 0 else "#2F7DD3" for value in values]

    fig = plt.figure(figsize=(10, 15), facecolor=BACKGROUND_COLOR)
    ax = fig.add_axes([0.34, 0.18, 0.62, 0.67])
    fig.patch.set_facecolor(BACKGROUND_COLOR)
    ax.set_facecolor(BACKGROUND_COLOR)
    bars = ax.barh(range(len(chart)), values, color=colors, height=0.62)
    y_positions = list(range(len(chart)))
    ax.set_yticks(y_positions)
    ax.set_yticklabels([])
    ax.grid(axis="x", color=GRID_COLOR, linewidth=0.9)
    ax.set_axisbelow(True)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.spines["bottom"].set_color(GRID_COLOR)
    ax.tick_params(axis="both", colors=TICK_COLOR, labelsize=TICK_SIZE, length=0)
    ax.set_xlabel(
        "전체 중앙값 대비 차이(%p)",
        fontsize=AXIS_TITLE_SIZE,
        color=TICK_COLOR,
        labelpad=14,
    )
    ax.xaxis.set_major_formatter(lambda value, _: f"{value:.0f}")
    limit = max(abs(values.min()), abs(values.max())) * 1.35
    ax.set_xlim(-limit, limit)
    ax.axvline(0, color=TICK_COLOR, linewidth=1.2)
    ax.text(
        0.01, 1.018, "← 전체 중앙값 미만", transform=ax.transAxes,
        ha="left", va="bottom", fontsize=13, color="#2F7DD3",
    )

    for y_position, (_, row) in zip(y_positions, chart.iterrows()):
        prefix = TextArea(
            (
                f"{int(row['준공연도'])}  "
                if order_by_completion_year
                else f"{int(row['순위'])}  "
            ),
            textprops={
                "color": TICK_COLOR, "fontsize": 13,
                "fontfamily": font_family, "fontweight": "bold",
            },
        )
        district = TextArea(
            str(row["자치구"]),
            textprops={
                "color": DISTRICT_COLORS[row["자치구"]],
                "fontsize": 13, "fontfamily": font_family,
                "fontweight": "bold",
            },
        )
        dong = TextArea(
            f"  {row['동']}",
            textprops={
                "color": TICK_COLOR, "fontsize": 13,
                "fontfamily": font_family, "fontweight": "bold",
            },
        )
        apartment_name = str(row["아파트"]).replace("(e편한세상)", "")
        completion_year_suffix = (
            ""
            if order_by_completion_year
            else f" '{int(row['준공연도']) % 100:02d}"
        )
        apartment = TextArea(
            f"  ·  {apartment_name}{completion_year_suffix}",
            textprops={
                "color": TICK_COLOR, "fontsize": 13,
                "fontfamily": font_family, "fontweight": "bold",
            },
        )
        packed_label = HPacker(
            children=[prefix, district, dong, apartment],
            align="center", pad=0, sep=0,
        )
        label_box = AnnotationBbox(
            packed_label, (0, y_position), xycoords=("axes fraction", "data"),
            xybox=(-247 if order_by_completion_year else -258, 0),
            boxcoords="offset points",
            box_alignment=(0, 0.5), frameon=False, pad=0,
        )
        ax.add_artist(label_box)
    ax.text(
        0.99, 1.018, "전체 중앙값 초과 →", transform=ax.transAxes,
        ha="right", va="bottom", fontsize=13, color="#F06432",
    )

    offset = limit * 0.02
    for bar, value, annual_growth in zip(
        bars, values, chart["annual_growth"]
    ):
        ax.text(
            value + (offset if value >= 0 else -offset),
            bar.get_y() + bar.get_height() / 2,
            f"{value:+.2f}%p ({annual_growth:.2%})",
            va="center",
            ha="left" if value >= 0 else "right",
            fontsize=DATA_LABEL_SIZE,
            fontweight="bold",
            color=bar.get_facecolor(),
        )

    fig.text(
        -0.02,
        0.972,
        "대도시 연구실",
        fontsize=BRAND_SIZE,
        color=SECONDARY_TEXT_COLOR,
        va="top",
    )
    fig.text(
        -0.02,
        0.944,
        (
            "강남3구 동별 대장단지 상승률 | 준공연도순"
            if order_by_completion_year
            else "강남3구 동별 대장단지 상승률 | 최근 10년"
        ),
        fontsize=TITLE_SIZE,
        fontweight="bold",
        color=TEXT_COLOR,
        va="top",
    )
    fig.text(
        -0.02,
        0.914,
        "KB 매매 일반가 · 2016년 7월~2026년 7월 · 300세대 이상 · "
        "전체 중앙값 기준",
        fontsize=SUBTITLE_SIZE,
        color=SECONDARY_TEXT_COLOR,
        va="top",
    )
    fig.text(
        -0.02,
        0.11,
        f"※ 전체 중앙값은 분석 대상 {len(chart)}개 대장단지의 연평균 상승률 중앙값 "
        f"{chart['annual_growth'].median():.2%}\n"
        "※ 막대 라벨: 전체 중앙값 대비 차이(%p) (해당 단지 연평균 상승률)\n"
        + ("※ 준공연도 오름차순으로 배열\n" if order_by_completion_year else "")
        + "※ 대장단지는 전용 75~93㎡·84㎡ 최근접 타입·300세대 이상 "
        "순수 아파트 중 "
        "2016년 7월 ㎡당 KB 일반가가 동별로 가장 높은 단지",
        fontsize=FOOTNOTE_SIZE,
        color=SECONDARY_TEXT_COLOR,
        va="top",
        linespacing=1.22,
    )
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(
        output_path,
        dpi=200,
        bbox_inches="tight",
        facecolor=BACKGROUND_COLOR,
    )
    plt.close(fig)
    return output_path


def display_download_link(label: str, path: Path) -> None:
    """Colab 다운로드 버튼 또는 로컬 파일 링크를 표시한다."""
    if IS_COLAB:
        from google.colab import files
        import ipywidgets as widgets

        button = widgets.Button(description=f"{label} 다운로드", icon="download")
        button.on_click(lambda _: files.download(str(path)))
        display(button)
    else:
        display(FileLink(str(path), result_html_prefix=f"{label}: "))


def main() -> None:
    """저장 자료를 읽어 동별 대장단지 표·그래프·CSV를 생성한다."""
    matched_types = load_matched_types(MATCHED_TYPE_PATH)
    flagships, complexes = calculate_flagships(matched_types)
    rank_comparison = calculate_rank_comparison(matched_types, flagships)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    flagships.to_csv(RESULT_PATH, index=False, encoding="utf-8-sig")
    create_chart(flagships, CHART_PATH)
    create_chart(
        flagships, COMPLETION_YEAR_CHART_PATH, order_by_completion_year=True
    )

    display(Markdown(build_criteria_markdown(matched_types, complexes, flagships)))
    display(HTML(build_table_html(flagships)))
    display(HTML(build_rank_comparison_table_html(rank_comparison)))
    display(Image(filename=str(CHART_PATH)))
    display(Image(filename=str(COMPLETION_YEAR_CHART_PATH)))
    display_download_link("결과 CSV", RESULT_PATH)
    display_download_link("그래프 PNG", CHART_PATH)
    display_download_link("준공연도순 그래프 PNG", COMPLETION_YEAR_CHART_PATH)


main()
